# NLG Narrative-Query Tracer (Project 2)

The real nlg system never stores the SQL query it uses to fetch data for each
`insight_type`'s narrative -- it builds the query, runs it, generates the
narrative text, and stores only the narrative (per insight_type, per record,
in Greenplum). **This notebook recovers that query using the exact same logic
nlg itself uses** -- it actually imports and runs the real
`fetch_data_with_insights_for_cl` / `_for_fl` / `_for_pr` function from
`nlg-src`, against a stub database connector that intercepts every SQL
statement instead of hitting a real database, and captures exactly what the
real code built.

**This is not an LLM approximation.** It's the real function, running with
real resolved config (table/schema names from `config/nlg_db_tables.ini`,
column mappings from `config/input_data_config/<target_type>.yml`, etc.). An
LLM-based reconstruction is kept only as a fallback for the rare case where
real execution can't run in your environment at all.

This is a **separate project** from `nlg_dag_agent`, though it reuses its
source loading/indexing. Make sure `nlg_dag_agent/` is importable alongside
`nlg_query_tracer/`.

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath("."))   # folder containing both nlg_dag_agent/ and nlg_query_tracer/

from nlg_dag_agent import Settings, build_hierarchy
from nlg_query_tracer import trace_target_type, trace_insight_type, trace_many_target_types, export_xlsx
from nlg_query_tracer.context_gatherer import list_insight_types_for_target_type, gather_context
from nlg_query_tracer.query_capture import capture_queries_for_insight_type

## 1. Build the source index

Reuses `nlg_dag_agent.build_hierarchy()`. An LLM provider is optional here --
only used as a fallback if real code execution fails in your environment.

In [ ]:
USE_LOCAL = False   # keep False -- connect to GitLab only, never read a local checkout

settings = Settings()
if USE_LOCAL:
    settings.local_dags_dir = "./nlg-dag/dags"
    settings.local_src_dir  = "./nlg-src/src"
else:
    settings.local_dags_dir = None   # forces GitLab mode in build_repos()
    settings.local_src_dir  = None
    # GITLAB_URL / NLG_PROJECT_PATH / NLG_DAGS_SUBPATH / NLG_SRC_SUBPATH already
    # default correctly for this repo layout (see nlg_dag_agent/config.py) --
    # override via env vars only if your project path differs.
settings.llm_provider = "azure_openai"   # optional fallback only -- set "none" to disable entirely
settings.resolve_secrets(interactive=True)   # prompts for the GitLab token (and LLM key, if needed)

hierarchy = build_hierarchy(settings)
print(f"Indexed {len(hierarchy.dags)} DAGs / source ready.")

## 2. Important: run this from an environment with nlg-src's real dependencies

Real execution imports the actual `nlg-src` code, so it needs whatever
third-party packages that code needs at import time (the exact same set
already installed wherever the real DAG runs -- e.g. `psycopg2`, and
whatever your `utils_narratives.py` import chain pulls in). Missing
*unrelated* imports (pulled in by other functions in the same file, never
actually called here) are auto-stubbed automatically and reported in
`warnings` -- but if the query itself looks wrong, install the real package
and re-run to rule that out.

## 3. See which insight_types exist for a target_type

In [ ]:
target_type = "account_holding_mmf"
insight_types = list_insight_types_for_target_type(hierarchy, target_type)

if not insight_types:
    print(f"No insight_types found for target_type='{target_type}'.")
    print(f"  -> list_insight_types_for_target_type() looks for rules/{target_type}/*.yml")
    print(f"     in your indexed nlg-src. Check that this folder exists and that")
    print(f"     target_type is spelled exactly as it appears there (case-sensitive),")
    print(f"     e.g. 'account_holding_mmf', 'account_sbl', 'fl_ubs_fa'.")
else:
    print(f"{len(insight_types)} insight_type(s) found for '{target_type}':")
insight_types

## 4. Inspect gathered context for one insight_type (no execution yet)

Useful to sanity-check the resolved config before running anything.

In [ ]:
if not insight_types:
    raise ValueError(
        f"No insight_types found for target_type='{target_type}' -- see the previous "
        f"cell's message for how to check/fix this before continuing."
    )

ctx = gather_context(hierarchy, target_type, insight_types[0])
print("variant function:", ctx.variant_function)
print("insights_target_user_type:", ctx.insights_target_user_type)
print("nlg_db_tables (from config/nlg_db_tables.ini):", {k: ctx.nlg_db_tables[k] for k in list(ctx.nlg_db_tables)[:5]}, "...")
print("source files gathered:")
for f in ctx.source_files:
    print(" -", f)
print("warnings:", ctx.warnings)

## 5. Run the real code and capture the query for one insight_type

`capture_queries_for_insight_type` is the low-level call: it returns every
SQL statement the real code issued (`all_captured`), plus which one is the
actual narrative-data query (`primary_query`).

In [ ]:
if not insight_types:
    raise ValueError(
        f"No insight_types found for target_type='{target_type}' -- see cell 6's "
        f"message for how to check/fix this before continuing."
    )

capture = capture_queries_for_insight_type(hierarchy, target_type, insight_types[0])
print("success:", capture.success, "| variant:", capture.variant_function)
if capture.execution_error:
    print("execution_error:", capture.execution_error)
print("\nall captured SQL statements:")
for c in capture.all_captured:
    print(f"  [{c['kind']}]", c['sql'][:100].replace(chr(10), ' '))
print("\n=== PRIMARY (narrative-data) QUERY ===\n")
print(capture.primary_query)

## 6. Trace every insight_type under a target_type, export to xlsx

`trace_insight_type`/`trace_target_type` wrap the above into the
`QueryResult` shape used by the xlsx export, and automatically fall back to
an LLM reconstruction only if real execution didn't produce a usable query.

In [ ]:
results = trace_target_type(hierarchy, target_type, settings=settings)
for r in results:
    print(r.target_type, "|", r.insight_type, "|", r.resolution_method)

out_path = export_xlsx(results, "./nlg_query_tracer_output/narrative_queries.xlsx")
print("\nSaved:", out_path, f"({len(results)} rows)")

## 7. Trace multiple target_types in one workbook

In [ ]:
results_all = trace_many_target_types(hierarchy, ["account_holding_mmf", "account_sbl", "fl_ubs_fa"], settings=settings)
export_xlsx(results_all, "./nlg_query_tracer_output/narrative_queries_multi.xlsx")

n_exec = sum(1 for r in results_all if r.resolution_method == "code_execution")
n_llm = sum(1 for r in results_all if r.resolution_method == "llm_fallback")
n_none = sum(1 for r in results_all if r.resolution_method == "none")
print(f"{len(results_all)} rows: code_execution={n_exec}, llm_fallback={n_llm}, none={n_none}")

## Notes / known limitations

- **`resolution_method` tells you how each row was produced.** `code_execution`
  = the real code actually ran and this is its real output (trust it).
  `llm_fallback` = real execution failed in this environment and an LLM
  reconstructed a best guess instead -- always verify those. `none` = both
  failed; check `notes`/`warnings` for why.
- **`--odm-env` / `odm_env=` picks the environment section of
  `config/nlg_db_tables.ini`** (default `prod_active`) for table/schema
  names -- switch it if you want the query as it would run in a different
  environment (`dev_active`, `dev_test`, ...).
- **Auto-stubbing** replaces any third-party import the *file* needs (but
  the *function we call* doesn't actually use) with a permissive fake, so
  import failures unrelated to query-building don't block execution. This
  is reported per-row in `warnings` for transparency.
- Extend `context_gatherer.VARIANT_BY_USER_TYPE` / `HELPER_FUNCTIONS` /
  `query_capture._candidate_kwargs` if your codebase adds new
  `fetch_data_with_insights_*` variants or new required parameters.
- `is_prospect_run=True` switches to the `fetch_data_with_insights_for_pr`
  code path.